In [ ]:
# Cell 1: Imports & Paths Setup
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add root directory to path
sys.path.append(os.path.abspath(".."))
import config
from src.preprocessing.lc_encoder import load_raw_record, level_crossing_encode

# Cell 2: Load Record 100 Signal
rec_path = os.path.join(config.DATA_RAW_DIR, "100")
signal, r_peaks, labels = load_raw_record(rec_path)

# Slice a 3-second segment (360 Hz * 3 = 1080 samples)
sample_window = 1080
ecg_segment = signal[:sample_window, :]

print(f"Loaded Record 100 segment with shape: {ecg_segment.shape}")

# Cell 3: Level-Crossing Encoding
spikes = level_crossing_encode(ecg_segment, delta_v=config.DELTA_V)
time_axis = np.arange(sample_window) / config.SAMPLING_RATE

# Cell 4: Plot Signal vs 4-Channel Binary Spikes
fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# Lead 1 Continuous ECG
axs[0].plot(time_axis, ecg_segment[:, 0], color='navy', label='Lead 1 ECG (mV)')
axs[0].set_ylabel('Amplitude (mV)')
axs[0].set_title('Raw Continuous ECG vs 1-Bit Event Spike Encodings')
axs[0].legend(loc='upper right')
axs[0].grid(True, alpha=0.3)

# Lead 1 Positive & Negative Spikes
axs[1].eventplot(np.where(spikes[:, 0] > 0)[0] / config.SAMPLING_RATE, lineoffsets=1, colors='crimson', label='Lead 1 + Spike')
axs[1].eventplot(np.where(spikes[:, 1] > 0)[0] / config.SAMPLING_RATE, lineoffsets=-1, colors='darkorange', label='Lead 1 - Spike')
axs[1].set_ylabel('Lead 1 Events')
axs[1].legend(loc='upper right')
axs[1].grid(True, alpha=0.3)

# Lead 2 Positive & Negative Spikes
axs[2].eventplot(np.where(spikes[:, 2] > 0)[0] / config.SAMPLING_RATE, lineoffsets=1, colors='seagreen', label='Lead 2 + Spike')
axs[2].eventplot(np.where(spikes[:, 3] > 0)[0] / config.SAMPLING_RATE, lineoffsets=-1, colors='purple', label='Lead 2 - Spike')
axs[2].set_ylabel('Lead 2 Events')
axs[2].set_xlabel('Time (seconds)')
axs[2].legend(loc='upper right')
axs[2].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(config.PLOTS_DIR, exist_ok=True)
plt.savefig(os.path.join(config.PLOTS_DIR, "lc_spike_raster.png"), dpi=300)
plt.show()

# Compression ratio calculation
raw_bits = ecg_segment.size * 32  # 32-bit floats
spike_bits = np.count_nonzero(spikes) * 1  # 1-bit events
print(f"Data Compression Factor: {raw_bits / spike_bits:.2f}x Reduction")